# Captioning as a Text Bridge — Connecting Vision to Language Pipelines

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/09_captioning_as_a_text_bridge.ipynb)

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

1. **Explain** the caption-as-bridge architecture and why it lets you reuse text-only infrastructure for visual data.
2. **Generate** image captions at different detail levels (brief, detailed, structured JSON) using prompt engineering with a quantized VLM.
3. **Build** a caption-first RAG pipeline that embeds captions with sentence-transformers and retrieves them with FAISS.
4. **Evaluate** caption quality using word-overlap metrics and an LLM-as-judge approach.
5. **Compare** the caption-then-query strategy against direct VLM querying and reason about the trade-offs.

## Section 1 — The Bridge Concept

Most production NLP stacks—retrieval-augmented generation, semantic search, summarisation,
entity extraction—are **text-only**. They ingest strings, embed strings, and generate strings.
Images live in a different modality, and plugging them into those stacks normally requires
expensive re-architecture.

**Captioning as a text bridge** sidesteps that problem entirely. The idea is deceptively
simple: run every image through a vision-language model once and produce a natural-language
description. From that point on the description is just another text chunk—your existing
embedder, vector store, and language model can consume it unchanged.

The pipeline looks like this:

```
Image  ──▶  VLM (caption)  ──▶  Text Description  ──▶  [ Embed / Index / Retrieve / Summarise ]
```

This architecture has three practical advantages:

| Advantage | Why it matters |
|---|---|
| **Infrastructure reuse** | You keep your existing text pipeline; no multimodal embedder needed. |
| **Interpretability** | Captions are human-readable, so you can inspect what the system "saw". |
| **Offline processing** | Captioning can run in batch; at query time only text is involved, so latency drops. |

The cost is **information loss**—a caption is a lossy compression of the pixels. We will
explore exactly when that loss matters in Sections 5 and 6.

In [ ]:
!pip install -q transformers torch qwen-vl-utils sentence-transformers faiss-cpu pillow bitsandbytes accelerate

In [ ]:
import json, time, textwrap
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

print(f"PyTorch {torch.__version__}  •  CUDA available: {torch.cuda.is_available()}")

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

VLM_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

processor = AutoProcessor.from_pretrained(VLM_ID)
vlm = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_ID, device_map="auto",
    quantization_config=bnb_config, torch_dtype="auto",
)
print(f"Loaded {VLM_ID}  •  GPU mem: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
from qwen_vl_utils import process_vision_info

def generate_caption(image, prompt, max_tokens=512):
    """Send an image + prompt to the VLM and return the generated text."""
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text",  "text": prompt},
    ]}]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(vlm.device)
    with torch.no_grad():
        output_ids = vlm.generate(**inputs, max_new_tokens=max_tokens)
    return processor.batch_decode(
        output_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )[0]

print("generate_caption() ready.")

### Creating synthetic test images

We generate simple images programmatically so the notebook is self-contained and does
not depend on external URLs.

In [ ]:
def make_park_scene():
    """Synthetic park scene with sky, grass, tree, sun, and bench."""
    img = Image.new("RGB", (640, 480), (135, 206, 235))
    draw = ImageDraw.Draw(img)
    draw.rectangle([0, 300, 640, 480], fill=(34, 139, 34))  # grass
    draw.rectangle([280, 160, 320, 320], fill=(139, 69, 19))  # trunk
    draw.ellipse([230, 80, 370, 200], fill=(0, 100, 0))  # canopy
    draw.ellipse([500, 30, 580, 110], fill=(255, 223, 0))  # sun
    draw.rectangle([400, 310, 500, 330], fill=(139, 90, 43))  # bench seat
    draw.rectangle([405, 330, 415, 360], fill=(100, 60, 30))  # leg L
    draw.rectangle([485, 330, 495, 360], fill=(100, 60, 30))  # leg R
    return img

def make_chart_image():
    """Synthetic bar chart with specific numerical values."""
    img = Image.new("RGB", (640, 480), (255, 255, 255))
    draw = ImageDraw.Draw(img)
    bars = [("Q1", 120, "steelblue"), ("Q2", 85, "coral"),
            ("Q3", 200, "seagreen"), ("Q4", 150, "goldenrod")]
    x = 80
    for label, h, color in bars:
        y_top = 400 - h
        draw.rectangle([x, y_top, x + 100, 400], fill=color)
        draw.text((x + 35, 410), label, fill="black")
        draw.text((x + 35, y_top - 20), str(h), fill="black")
        x += 130
    draw.text((200, 20), "Quarterly Revenue ($k)", fill="black")
    return img

def make_abstract_image():
    """Random coloured circles — intentionally abstract."""
    img = Image.new("RGB", (640, 480), (20, 20, 30))
    draw = ImageDraw.Draw(img)
    rng = np.random.default_rng(42)
    for _ in range(30):
        cx, cy = rng.integers(0, 640), rng.integers(0, 480)
        r = int(rng.integers(15, 60))
        color = tuple(rng.integers(80, 255, size=3).tolist())
        draw.ellipse([cx - r, cy - r, cx + r, cy + r], fill=color)
    return img

park_img   = make_park_scene()
chart_img  = make_chart_image()
abstract_img = make_abstract_image()

display(park_img)
display(chart_img)
display(abstract_img)
print("Three test images created.")

In [ ]:
caption = generate_caption(park_img, "Describe this image in one sentence.")
print("Park caption:", caption)

## Section 2 — Caption Quality Spectrum

Not all captions are created equal. A single image can produce wildly different text
depending on the prompt you send to the VLM. Broadly, captions fall along a spectrum of
detail:

| Level | Typical length | Best for |
|---|---|---|
| **Brief** | 10-20 words | Thumbnails, quick search labels |
| **Detailed** | 50-150 words | RAG chunks, accessibility alt-text |
| **Structured** | JSON / key-value | Metadata extraction, downstream parsing |

The key insight is that **the prompt is the knob** that controls where on this spectrum
you land. A terse instruction ("Describe briefly.") yields a short label; a detailed
instruction ("Give a comprehensive description including spatial layout, colours, objects,
and mood.") produces a paragraph; and a schema-guided instruction ("Return JSON with keys
objects, colours, mood.") yields structured output.

Choosing the right level depends on your downstream task. For semantic search, brief
captions may suffice because the embedding model compresses them anyway. For question
answering, detailed captions preserve more information that the reader model can draw on.
For analytics pipelines, structured JSON lets you skip NLP parsing entirely.

In [ ]:
BRIEF_PROMPT = "Describe this image in a single short sentence."
brief_caption = generate_caption(park_img, BRIEF_PROMPT)
print(f"[BRIEF]  ({len(brief_caption.split())} words)")
print(brief_caption)

In [ ]:
DETAIL_PROMPT = (
    "Give a comprehensive description of this image. Include the spatial layout, "
    "colours, objects, and the overall mood or setting."
)
detailed_caption = generate_caption(park_img, DETAIL_PROMPT)
print(f"[DETAILED]  ({len(detailed_caption.split())} words)")
print(textwrap.fill(detailed_caption, 90))

In [ ]:
STRUCT_PROMPT = (
    "Describe this image as a JSON object with the following keys: "
    "\"objects\" (list of strings), \"dominant_colours\" (list of strings), "
    "\"scene_type\" (string), \"mood\" (string). Return only the JSON."
)
structured_caption = generate_caption(park_img, STRUCT_PROMPT)
print(f"[STRUCTURED]  ({len(structured_caption.split())} words)")
print(structured_caption)

In [ ]:
rows = [
    ("Brief",      brief_caption),
    ("Detailed",   detailed_caption),
    ("Structured", structured_caption),
]
print(f"{'Level':<12} {'Words':>6}  {'Chars':>6}")
print("-" * 30)
for level, cap in rows:
    print(f"{level:<12} {len(cap.split()):>6}  {len(cap):>6}")

## Section 3 — Captioning for RAG

Retrieval-Augmented Generation (RAG) is the dominant pattern for grounding LLM answers in
external knowledge. The standard flow is: split documents into text chunks, embed them,
store embeddings in a vector index, and at query time retrieve the top-*k* chunks to
provide context to the language model.

Images break this flow because a text embedder cannot consume pixels. The caption-first
strategy restores compatibility: we caption each image, and the resulting text becomes just
another chunk in the index.

```
Images ──▶ VLM captions ──┐
                          ├──▶ Sentence-Transformer ──▶ FAISS index ──▶ Retriever
Text chunks ──────────────┘
```

Below we build a minimal but complete caption-based RAG pipeline. We embed captions and
plain-text chunks into the same FAISS index, then query it.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(f"Embedder loaded — dim {embedder.get_sentence_embedding_dimension()}")

In [ ]:
# Generate captions for all images
CAPTION_PROMPT = "Describe this image in detail."

image_captions = [
    ("park",     generate_caption(park_img,     CAPTION_PROMPT)),
    ("chart",    generate_caption(chart_img,    CAPTION_PROMPT)),
    ("abstract", generate_caption(abstract_img, CAPTION_PROMPT)),
]

# Plain-text knowledge chunks
text_chunks = [
    "The quarterly revenue report shows Q3 as the strongest quarter with $200k.",
    "Urban parks improve mental health and provide green space for exercise.",
    "Abstract art uses shapes, colours, and forms to achieve its effect.",
    "FAISS is a library for efficient similarity search developed by Meta.",
]

# Combine captions and text chunks
all_docs = [cap for _, cap in image_captions] + text_chunks
doc_labels = [f"img:{name}" for name, _ in image_captions] + \
             [f"txt:{i}" for i in range(len(text_chunks))]

print(f"Corpus size: {len(all_docs)} documents ({len(image_captions)} captions + {len(text_chunks)} text chunks)")
for lbl, doc in zip(doc_labels, all_docs):
    print(f"  [{lbl}] {doc[:80]}..." if len(doc) > 80 else f"  [{lbl}] {doc}")

In [ ]:
embeddings = embedder.encode(all_docs, normalize_embeddings=True).astype(np.float32)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"FAISS index built — {index.ntotal} vectors of dim {dim}")

In [ ]:
queries = [
    "What does the park scene look like?",
    "Which quarter had the highest revenue?",
    "Tell me about abstract art.",
]

for q in queries:
    q_emb = embedder.encode([q], normalize_embeddings=True).astype(np.float32)
    scores, idxs = index.search(q_emb, k=3)
    print(f"\nQuery: {q}")
    for rank, (score, idx) in enumerate(zip(scores[0], idxs[0]), 1):
        print(f"  #{rank}  [{doc_labels[idx]}]  score={score:.3f}")
        print(f"        {all_docs[idx][:100]}")

## Section 4 — Caption Quality Evaluation

A caption is only useful if it faithfully represents the image. In production you need
automatic ways to assess caption quality along three axes:

* **Faithfulness** — Does the caption describe things that are actually in the image?
* **Completeness** — Does the caption mention every important element?
* **Hallucination** — Does the caption invent details that are *not* present?

Classic metrics like BLEU and CIDEr compare a generated caption against human-written
references. When references are unavailable (the common case in production), we can fall
back to two lightweight strategies:

1. **Word-overlap with a ground-truth object list** — check whether expected object
   names appear in the generated caption.
2. **LLM-as-judge** — ask a language model to rate the caption on a rubric, using the
   image (or a known description) as the reference.

Below we implement both approaches.

In [ ]:
def word_overlap_score(caption: str, expected_objects: list[str]) -> dict:
    """Compute recall and precision of expected objects in the caption."""
    caption_lower = caption.lower()
    found = [obj for obj in expected_objects if obj.lower() in caption_lower]
    recall    = len(found) / len(expected_objects) if expected_objects else 0.0
    precision = len(found) / len(caption_lower.split()) if caption_lower.split() else 0.0
    return {"found": found, "missed": [o for o in expected_objects if o not in found],
            "recall": recall, "precision": precision}

# The park image should contain: sky, grass, tree, sun, bench
park_objects = ["sky", "grass", "tree", "sun", "bench"]
result = word_overlap_score(detailed_caption, park_objects)
print(f"Caption: {detailed_caption[:100]}...")
print(f"Expected objects : {park_objects}")
print(f"Found            : {result['found']}")
print(f"Missed           : {result['missed']}")
print(f"Object recall    : {result['recall']:.2f}")

In [ ]:
# --- LLM-as-judge evaluation ---
# We ask the VLM itself to rate a caption on faithfulness and completeness.

JUDGE_PROMPT = (
    "You are an image-caption evaluator. You are given an image and a candidate caption.\n"
    "Rate the caption on two criteria (1-5 scale):\n"
    "  - Faithfulness: Does the caption only describe things present in the image?\n"
    "  - Completeness: Does the caption mention all major visible elements?\n"
    "Return ONLY a JSON object: {\"faithfulness\": <int>, \"completeness\": <int>, "
    "\"explanation\": <string>}\n\n"
    "Caption to evaluate:\n"
)

judge_input = JUDGE_PROMPT + detailed_caption
judge_output = generate_caption(park_img, judge_input, max_tokens=256)
print("Judge response:")
print(judge_output)

## Section 5 — When Captioning Fails

The caption bridge is powerful but not perfect. Three families of images are
particularly problematic:

1. **Diagrams and charts with precise values.** A bar chart labelled "$200 k" may be
   captioned as "a bar chart showing quarterly revenue" — the exact numbers often vanish.
   If your downstream task needs those numbers, captioning alone is insufficient; you need
   OCR or structured extraction.

2. **Abstract or artistic images.** When there are no recognisable objects, the model
   resorts to vague language ("colourful shapes on a dark background"), which produces
   low-information embeddings that are hard to distinguish from one another.

3. **Dense scenes with many objects.** A photograph of a crowded marketplace may contain
   dozens of relevant items. Captions tend to mention only the most salient ones, losing
   the long tail of small but important details.

Below we demonstrate the chart and abstract failure modes using our synthetic images.

In [ ]:
# Chart image — does the caption preserve the exact numbers?
chart_caption = generate_caption(
    chart_img,
    "Describe this chart in detail, including all numerical values.",
)
print("=== Chart caption ===")
print(textwrap.fill(chart_caption, 90))

expected_numbers = ["120", "85", "200", "150"]
found = [n for n in expected_numbers if n in chart_caption]
print(f"\nExpected values: {expected_numbers}")
print(f"Found in caption: {found}")
print(f"Value recall: {len(found)/len(expected_numbers):.0%}")

In [ ]:
# Abstract image — is the caption informative enough to distinguish it?
abstract_caption = generate_caption(
    abstract_img,
    "Describe this image in detail.",
)
print("=== Abstract caption ===")
print(textwrap.fill(abstract_caption, 90))
print(f"\nWord count: {len(abstract_caption.split())}")
print("Note: abstract images tend to produce generic, low-information captions.")

## Section 6 — Bridge vs Direct VLM Querying

There are two fundamentally different ways to answer a question about an image with a
language model:

| Strategy | Flow | When to prefer |
|---|---|---|
| **Caption-then-query** | Image → VLM caption → text LLM answers from caption | Batch indexing, offline processing, cost-sensitive workloads |
| **Direct VLM query** | Image + question → VLM answers directly | One-off questions, accuracy-critical tasks, visual detail needed |

The caption-then-query approach runs the VLM once to produce a caption, then feeds that
caption (text only) into a cheaper text LLM for question answering. This is faster at
query time—especially if captions are pre-computed—but the answer can only reference
information that survived the captioning step.

Direct querying sends the raw image to the VLM along with the question, giving the model
full access to every pixel. It is more accurate for fine-grained visual questions but
incurs higher latency and cost per query.

Below we compare both strategies on the same set of questions.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

TEXT_LLM_ID = "Qwen/Qwen3-8B"

text_tokenizer = AutoTokenizer.from_pretrained(TEXT_LLM_ID)
text_llm = AutoModelForCausalLM.from_pretrained(
    TEXT_LLM_ID, device_map="auto",
    quantization_config=bnb_config, torch_dtype="auto",
)
print(f"Loaded {TEXT_LLM_ID}  •  GPU mem: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
def ask_text_llm(prompt, max_tokens=256):
    """Generate a response from the text-only LLM."""
    messages = [{"role": "user", "content": prompt}]
    text = text_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = text_tokenizer(text, return_tensors="pt").to(text_llm.device)
    with torch.no_grad():
        out = text_llm.generate(**inputs, max_new_tokens=max_tokens)
    return text_tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("ask_text_llm() ready.")

In [ ]:
# Pre-compute a detailed caption for the park image
park_detail = generate_caption(park_img, "Describe this image in detail.")

questions = [
    "What colour is the sky in the image?",
    "Is there a bench in the scene?",
    "How many trees are visible?",
]

print(f"Pre-computed caption:\n  {park_detail}\n")
print("=" * 80)

for q in questions:
    # Strategy A: caption-then-query
    t0 = time.time()
    bridge_prompt = f"Based on this description of an image:\n{park_detail}\n\nAnswer: {q}"
    bridge_answer = ask_text_llm(bridge_prompt)
    bridge_time = time.time() - t0

    # Strategy B: direct VLM query
    t0 = time.time()
    direct_answer = generate_caption(park_img, q, max_tokens=128)
    direct_time = time.time() - t0

    print(f"\nQ: {q}")
    print(f"  [Bridge]  ({bridge_time:.1f}s)  {bridge_answer.strip()[:120]}")
    print(f"  [Direct]  ({direct_time:.1f}s)  {direct_answer.strip()[:120]}")

## Exercises

The following exercises reinforce the concepts from this notebook. Starter code is
provided; fill in the parts marked `# TODO`.

### Exercise 1 — Multi-image captioning pipeline

Write a function that accepts a list of PIL images, captions each one, embeds the
captions, and returns a ready-to-query FAISS index together with the caption list.

In [ ]:
def build_caption_index(images, caption_prompt="Describe this image in detail."):
    """Caption a list of images and return (index, captions)."""
    captions = []
    for img in images:
        # TODO: generate a caption for each image
        cap = ""  # replace with generate_caption(...)
        captions.append(cap)

    # TODO: embed captions and build a FAISS index
    index = None  # replace
    return index, captions

# Test with our three images
# idx, caps = build_caption_index([park_img, chart_img, abstract_img])

### Exercise 2 — Caption granularity sweep

Design an experiment that generates captions at five different detail levels for the
same image and measures how retrieval quality (top-1 score) changes as captions get
longer.

In [ ]:
detail_prompts = [
    "Name the main object.",
    "Describe the image in one sentence.",
    "Describe the image in a short paragraph.",
    "Give a comprehensive description including layout, colours, and mood.",
    "Give an exhaustive description covering every visible element, its position, "
    "colour, size, and relation to other elements.",
]

# TODO: for each prompt, generate a caption, embed it, and measure cosine
# similarity against a reference query like "a peaceful outdoor scene".
# Plot word count (x) vs similarity score (y).

### Exercise 3 — Hallucination detector

Implement a function that uses the VLM-as-judge pattern to detect hallucinated objects.
Given an image and a caption, the function should return a list of objects mentioned in
the caption that are *not* visible in the image.

In [ ]:
def detect_hallucinations(image, caption):
    """Return a list of hallucinated objects mentioned in the caption."""
    prompt = (
        f"A caption was generated for the attached image:\n"
        f"\"{caption}\"\n\n"
        f"List any objects or details mentioned in the caption that are NOT "
        f"visible in the image. Return a JSON list of strings. "
        f"If nothing is hallucinated, return an empty list []."
    )
    # TODO: call generate_caption(image, prompt) and parse the JSON list
    return []

# Test: feed a caption that deliberately includes a hallucinated object
# test_caption = detailed_caption + " A red car is parked near the bench."
# detect_hallucinations(park_img, test_caption)

## Key Takeaways

| # | Insight |
|---|---|
| 1 | Captioning converts images into text so they can enter any text-only pipeline without infrastructure changes. |
| 2 | The **prompt** controls caption detail — brief labels, rich paragraphs, or structured JSON are all achievable with the same model. |
| 3 | Caption-first RAG lets you embed images and text chunks into a **single FAISS index** and retrieve them uniformly. |
| 4 | Caption quality can be measured with word-overlap recall (cheap) or LLM-as-judge scoring (more nuanced). |
| 5 | Charts, abstract art, and dense scenes are the main **failure modes** — exact values and low-salience details get lost. |
| 6 | Direct VLM querying preserves more visual information but costs more latency; choose the strategy based on your accuracy and cost constraints. |

## References

1. Wang, P. et al. (2024). *Qwen2.5-VL Technical Report*. arXiv:2502.13923.
2. Reimers, N. & Gurevych, I. (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks*. EMNLP.
3. Johnson, J. et al. (2021). *Billion-Scale Similarity Search with GPUs (FAISS)*. IEEE Transactions on Big Data.
4. Li, J. et al. (2023). *BLIP-2: Bootstrapping Language-Image Pre-training with Frozen Image Encoders and Large Language Models*. ICML.
5. Zheng, L. et al. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena*. NeurIPS.

---

⬅️ [08 — Small VLMs and Multi-Image Workflows](08_small_vlms_and_multi_image_workflows.ipynb)  
➡️ [10 — Page-as-Image Retrieval with ColPali](10_page_as_image_retrieval_with_colpali.ipynb)